In [1]:
!pip install datasets transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


Access the travel-related dataset directly from Hugging Face

In [15]:
from datasets import load_dataset

# Load the travel questions-response dataset
dataset = load_dataset("JasleenSingh91/travel-questions-response")

print(dataset['train'][0])  # View a sample


{'Input': 'Can you suggest the best travel destinations for my next trip?', 'Response_1': ' Absolutely! To help me suggest the best travel destinations for your next trip, I\'ll need to know a few things from you:\n1. What type of trip are you looking for? (e.g., beach vacation, adventure travel, city break, cultural tour, etc.)\n2. What is your preferred climate? (e.g., warm, cold, tropical, dry, etc.)\n3. How long will you be traveling for?\n4. What is your budget range?\n5. Do you have any specific travel preferences, such as accessibility for individuals with disabilities or dietary restrictions?\nBased on your answers to these questions, I can suggest some travel destinations that may fit your needs and preferences. Here are a few options to consider:\n1. Maui, Hawaii: If you\'re looking for a warm, tropical beach vacation, Maui is a great option. With beautiful beaches, fantastic weather, and a variety of activities, Maui has something for everyone.\n2. Queenstown, New Zealand: F

model select

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"  # A lightweight GPT-2 variant
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


Format the dataset for training

In [17]:
# Add a padding token if not already present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Use EOS token as padding

def preprocess_function(example):
    text = f"Question: {example['Input']} Answer: {example['Response_1']}"
    tokenized = tokenizer(text, padding="max_length", truncation=True)
    tokenized["labels"] = tokenized["input_ids"].copy()  # Use input_ids as labels
    return tokenized


tokenized_datasets = dataset.map(preprocess_function, remove_columns=dataset["train"].column_names)

# Tokenize the dataset
#tokenized_datasets = tokenized_datasets.map(
#    lambda x: tokenizer(x["text"], padding="max_length", truncation=True),
#    batched=True
#)




Map:   0%|          | 0/6009 [00:00<?, ? examples/s]

Fine-Tune the Model

In [28]:
from transformers import Trainer, TrainingArguments
import os
os.environ["WANDB_DISABLED"] = "true"

tokenized_datasets = tokenized_datasets["train"].train_test_split(test_size=0.2)
train_dataset = tokenized_datasets["train"]
val_dataset = tokenized_datasets["test"]

training_args = TrainingArguments(
    output_dir="./results",              # Directory to save results
    evaluation_strategy="epoch",         # Evaluate at the end of every epoch
    learning_rate=2e-5,                  # Learning rate
    per_device_train_batch_size=8,       # Batch size for training
    per_device_eval_batch_size=8,        # Batch size for evaluation
    num_train_epochs=3,                  # Number of epochs
    weight_decay=0.01,                   # Weight decay for optimizer
    save_total_limit=1,                  # Save only the latest checkpoint
    report_to="none",                    # Disable W&B logging
    logging_dir="./logs",                # Directory for logging
    logging_steps=100,                   # Log every 100 steps
)

# Initialize Trainer
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=tokenized_datasets["train"],  # Training dataset
    eval_dataset=tokenized_datasets["test"],   # Evaluation dataset
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [29]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.633800,0.606501
2,0.616400,0.599529
3,0.615400,0.597194


TrainOutput(global_step=1443, training_loss=0.6249721577327778, metrics={'train_runtime': 2880.835, 'train_samples_per_second': 4.004, 'train_steps_per_second': 0.501, 'total_flos': 3014058018078720.0, 'train_loss': 0.6249721577327778, 'epoch': 3.0})

Generate Responses

In [32]:
# Generate responses using the fine-tuned model
def generate_response(input_text, max_length=50):
    # Encode the input text
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    # Move input_ids to the same device as the model
    input_ids = input_ids.to(model.device)

    # Generate a response
    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True,  # Enable sampling for temperature to take effect
        pad_token_id=tokenizer.eos_token_id,  # Avoid warnings about padding
    )

    # Decode and return the response
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Test the model
input_text = "i want a 5 km route"
response = generate_response(input_text)
print("Generated Response:", response)


Generated Response: i want a 5 km route option for a winter adventure, with destinations like skiing, snowboarding, and cozy retreats. Answer: Absolutely, I'd be happy to help you plan a winter adventure that offers a great mix of skiing, snowboarding


Save and Reload the Model

In [26]:
# Save the model
model.save_pretrained("./my_travel_bot")
tokenizer.save_pretrained("./my_travel_bot")

# Reload the model
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("./my_travel_bot")
tokenizer = AutoTokenizer.from_pretrained("./my_travel_bot")
